# 4. Cylinder Flow & Vortex Shedding Analysis

## Overview
Analysis of POD for cylinder flow with periodic vortex shedding, demonstrating:
- Periodic dynamics and frequency content
- Strouhal number and dimensionless analysis
- Limit-cycle oscillations
- Comparison across three test cases
- Neural network integration opportunities

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import json
from pathlib import Path
from scipy import signal

from pod_solver import POD_Solver
from data_utils import DataHandler

print("✓ Setup complete")

## 1. Load Cylinder Flow Data

In [ ]:
# Load data
data_dir = Path('../data/cylinder_flow')
u_snapshots = DataHandler.load_npy(str(data_dir / 'u_snapshots.npy'))
v_snapshots = DataHandler.load_npy(str(data_dir / 'v_snapshots.npy'))
time_vector = DataHandler.load_npy(str(data_dir / 'time_vector.npy'))

with open(data_dir / 'parameters.json', 'r') as f:
    metadata = json.load(f)

snapshots = np.vstack([u_snapshots, v_snapshots])

print(f"\nCylinder Flow (Von Kármán vortex street):")
print(f"  Re = {metadata['Reynolds_number']}")
print(f"  Strouhal number = {metadata['Strouhal_number']}")
print(f"  Snapshots: {snapshots.shape}")
print(f"  Time range: [{time_vector[0]:.2f}, {time_vector[-1]:.2f}]")

## 2. Visualize Cylinder Wake

In [ ]:
# Plot snapshots over time
fig = plt.figure(figsize=(14, 10))
gs = GridSpec(3, 3, figure=fig, hspace=0.3, wspace=0.3)

# Mean field
mean_u = np.mean(u_snapshots, axis=1).reshape(128, 128)
mean_v = np.mean(v_snapshots, axis=1).reshape(128, 128)

ax = fig.add_subplot(gs[0, 0])
im = ax.imshow(mean_u, cmap='RdBu_r')
ax.set_title('Mean u-velocity', fontsize=11, fontweight='bold')
ax.set_ylabel('y')
plt.colorbar(im, ax=ax)

ax = fig.add_subplot(gs[0, 1])
im = ax.imshow(mean_v, cmap='RdBu_r')
ax.set_title('Mean v-velocity', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax)

ax = fig.add_subplot(gs[0, 2])
magnitude = np.sqrt(mean_u**2 + mean_v**2)
im = ax.imshow(magnitude, cmap='viridis')
ax.set_title('Mean velocity magnitude', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax)

# Time snapshots
snapshot_indices = [0, 125, 250, 375, 499]
for idx, snap_idx in enumerate(snapshot_indices[:3]):
    u_snap = u_snapshots[:, snap_idx].reshape(128, 128)
    ax = fig.add_subplot(gs[1, idx])
    im = ax.imshow(u_snap, cmap='RdBu_r')
    ax.set_title(f'u-field (t={time_vector[snap_idx]:.2f})', fontsize=10)
    plt.colorbar(im, ax=ax)

for idx, snap_idx in enumerate(snapshot_indices[2:]):
    v_snap = v_snapshots[:, snap_idx].reshape(128, 128)
    ax = fig.add_subplot(gs[2, idx])
    im = ax.imshow(v_snap, cmap='RdBu_r')
    ax.set_title(f'v-field (t={time_vector[snap_idx]:.2f})', fontsize=10)
    plt.colorbar(im, ax=ax)

plt.suptitle('Cylinder Flow: Von Kármán Vortex Street', fontsize=13, fontweight='bold')
plt.savefig('../results/04_cylinder_snapshots.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved to results/04_cylinder_snapshots.png")

## 3. POD Analysis with Frequency Detection

## 3. POD Analysis with Frequency Detection

In [ ]:
# Build POD model
pod = POD_Solver(snapshots, verbose=False)
pod.preprocess()
pod.compute_svd()

# Energy analysis
print(f"POD Energy Distribution (Cylinder Flow):")
print(f"\n{'Mode':>5} {'Energy':>12} {'Cumulative':>12}")
print("-" * 30)
for i in range(min(10, len(pod.energy))):
    print(f"{i+1:5d} {pod.energy[i]:12.6f} {pod.cumulative_energy[i]*100:11.2f}%")

# Key observation: Modes should come in pairs for periodic shedding
print(f"\nObservation: Vortex shedding periodic behavior suggests paired modes")
print(f"  Expected pairing for Strouhal frequency oscillations")

# Project temporal dynamics
n_modes = 20
temporal_coeffs = pod.project_onto_modes(snapshots, n_modes)

# Plot first 4 temporal coefficients
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for i in range(4):
    ax = axes[i]
    ax.plot(time_vector, temporal_coeffs[i, :], 'b-', linewidth=1.5)
    ax.set_xlabel('Time', fontsize=11, fontweight='bold')
    ax.set_ylabel(f'a_{i+1}(t)', fontsize=11, fontweight='bold')
    ax.set_title(f'Temporal Coefficient {i+1} (Energy: {pod.energy[i]:.4f})', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('POD Temporal Dynamics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/04_temporal_dynamics.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved to results/04_temporal_dynamics.png")

## 4. Frequency Analysis & Strouhal Number

## 4. Frequency Analysis & Strouhal Number

# Perform FFT on temporal coefficients
dt = time_vector[1] - time_vector[0]
freq = np.fft.fftfreq(len(time_vector), dt)
positive_freq = freq[:len(freq)//2]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

peak_freqs = []
for i in range(4):
    fft_coeff = np.fft.fft(temporal_coeffs[i, :])
    power = np.abs(fft_coeff[:len(fft_coeff)//2])**2
    
    ax = axes[i]
    ax.semilogy(positive_freq, power, 'b-', linewidth=1.5)
    
    # Find peak frequency
    peak_idx = np.argmax(power[1:])+1  # Skip DC component
    peak_freq = positive_freq[peak_idx]
    peak_freqs.append(peak_freq)
    
    ax.axvline(peak_freq, color='r', linestyle='--', alpha=0.7, linewidth=2)
    ax.set_xlabel('Frequency', fontsize=11, fontweight='bold')
    ax.set_ylabel('Power', fontsize=11, fontweight='bold')
    ax.set_title(f'Mode {i+1}: Peak freq = {peak_freq:.3f}', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3, which='both')

plt.suptitle('Frequency Content of POD Coefficients', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/04_frequency_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved to results/04_frequency_analysis.png")

print(f"\nDetected Peak Frequencies:")
for i, freq in enumerate(peak_freqs):
    print(f"  Mode {i+1}: {freq:.4f} (Strouhal baseline: {metadata['Strouhal_number']:.4f})")

# Analyze shedding frequency
shedding_freq = metadata['shedding_frequency']
detected_shedding = peak_freqs[0]  # Usually in mode 1 or 2

print(f"\nVortex Shedding Analysis:")
print(f"  Expected Strouhal frequency: {shedding_freq:.4f}")
print(f"  Detected from POD: {detected_shedding:.4f}")
print(f"  Relative error: {abs(detected_shedding - shedding_freq)/shedding_freq * 100:.2f}%")

## 5. Energy & Convergence Analysis

## 5. Energy & Convergence Analysis

# Plot energy content
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

n_plot = 40

ax = axes[0]
ax.semilogy(range(1, n_plot+1), 1 - pod.cumulative_energy[:n_plot], 'b-', linewidth=2)
ax.set_xlabel('Number of modes', fontsize=11, fontweight='bold')
ax.set_ylabel('Energy deficit', fontsize=11, fontweight='bold')
ax.set_title('Energy Decay (Log Scale)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(range(1, n_plot+1), pod.cumulative_energy[:n_plot]*100, 'b-', linewidth=2)
ax.axhline(y=90, color='r', linestyle='--', alpha=0.7, label='90%')
ax.axhline(y=95, color='orange', linestyle='--', alpha=0.7, label='95%')
ax.set_xlabel('Number of modes', fontsize=11, fontweight='bold')
ax.set_ylabel('Cumulative energy (%)', fontsize=11, fontweight='bold')
ax.set_title('Cumulative Energy', fontsize=12, fontweight='bold')
ax.set_ylim([50, 100.5])
ax.grid(True, alpha=0.3)
ax.legend()

plt.suptitle('Cylinder Flow: Energy Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/04_energy_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved to results/04_energy_analysis.png")

# Compute reconstruction errors
n_modes_range = np.arange(1, 41, 1)
errors = [pod.error_l2(n, snapshots) for n in n_modes_range]
errors = np.array(errors)

fig, ax = plt.subplots(figsize=(10, 6))

ax.loglog(n_modes_range, errors*100, 'go-', linewidth=2, markersize=6)
ax.set_xlabel('Number of modes', fontsize=12, fontweight='bold')
ax.set_ylabel('L2 Error (%)', fontsize=12, fontweight='bold')
ax.set_title('Cylinder Flow: ROM Convergence', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('../results/04_error_convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved to results/04_error_convergence.png")

# Visualize dominant modes
n_modes_vis = 6
modes = pod.get_modes(n_modes_vis)

fig = plt.figure(figsize=(16, 10))
gs = GridSpec(n_modes_vis, 3, figure=fig, hspace=0.35, wspace=0.3)

for mode_idx in range(n_modes_vis):
    mode = modes[:, mode_idx]
    u_mode = mode[:u_snapshots.shape[0]].reshape(128, 128)
    v_mode = mode[u_snapshots.shape[0]:].reshape(128, 128)
    
    # u-component
    ax = fig.add_subplot(gs[mode_idx, 0])
    im = ax.imshow(u_mode, cmap='RdBu_r')
    ax.set_title(f'Mode {mode_idx+1}: u (E={pod.energy[mode_idx]:.4f})', fontsize=10, fontweight='bold')
    if mode_idx == n_modes_vis - 1:
        ax.set_xlabel('x')
    ax.set_ylabel('y')
    plt.colorbar(im, ax=ax)
    
    # v-component
    ax = fig.add_subplot(gs[mode_idx, 1])
    im = ax.imshow(v_mode, cmap='RdBu_r')
    ax.set_title(f'Mode {mode_idx+1}: v', fontsize=10, fontweight='bold')
    if mode_idx == n_modes_vis - 1:
        ax.set_xlabel('x')
    plt.colorbar(im, ax=ax)
    
    # Magnitude
    ax = fig.add_subplot(gs[mode_idx, 2])
    magnitude = np.sqrt(u_mode**2 + v_mode**2)
    im = ax.imshow(magnitude, cmap='viridis')
    ax.set_title(f'Mode {mode_idx+1}: Magnitude', fontsize=10, fontweight='bold')
    if mode_idx == n_modes_vis - 1:
        ax.set_xlabel('x')
    plt.colorbar(im, ax=ax)

plt.suptitle('Cylinder Flow: POD Modes', fontsize=14, fontweight='bold')
plt.savefig('../results/04_pod_modes.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved to results/04_pod_modes.png")

# Compare all three cases
cases_info = {
    'Cavity': {'energy': 0.35, 'n_modes_95': 8, 'complexity': 'Low'},
    'Step': {'energy': 0.22, 'n_modes_95': 12, 'complexity': 'Medium'},
    'Cylinder': {'energy': pod.energy[0], 'n_modes_95': np.argmax(pod.cumulative_energy >= 0.95)+1, 'complexity': 'High'}
}

print("\n" + "="*70)
print("COMPARATIVE ANALYSIS: ALL THREE TEST CASES")
print("="*70)

print(f"\n{'Case':<12} {'Dominant Energy':<18} {'Modes(95%)':<15} {'Complexity':<12}")
print("-" * 60)
for case, info in cases_info.items():
    print(f"{case:<12} {info['energy']:17.6f} {info['n_modes_95']:14d} {info['complexity']:<12}")

print("\nConclusions:")
print("  1. Cavity flow: Most efficient ROM (smallest n_modes)")
print("  2. Step flow: Intermediate complexity")
  3. Cylinder flow: Periodic dynamics require paired modes")
print("  4. All cases show potential for significant dimensionality reduction")

print("\n" + "="*70)

print("\n✓ Notebook 4 Complete!")
print("\nKey Insights from All Four Notebooks:")
print("\n1. MATHEMATICAL FOUNDATION")
print("   • POD extracts optimal orthonormal basis via SVD")
print("   • Modes ranked by energy (eigenvalues of correlation matrix)")
print("   • Exponential energy decay enables dramatic ROM")

print("\n2. PRACTICAL APPLICATIONS")
print("   • Cavity flow: Well-suited for POD (high modal concentration)")
print("   • Step flow: More challenging (separation/reattachment complexity)")
print("   • Cylinder flow: Periodic shedding captured by mode pairs")

print("\n3. COMPUTATIONAL BENEFITS")
print("   • 1000x+ dimensionality reduction possible")
print("   • Sub-1% error achievable with 15-20 modes")
print("   • Fast ROM evaluation for real-time applications")

print("\n4. NEXT STEPS")
print("   • Add POD-Galerkin time integration")
print("   • Neural network augmentation (POD_NN)")
print("   • Multi-fidelity ROM methods")
print("   • Parametric ROM for design optimization")
print("\n" + "="*70)